In [1]:
import numpy as np

with open('GUTENBERG.txt', 'r', encoding = "utf8") as fp:
  text = fp.read()
start_idx = text.find("THE MYSTERIOUS ISLAND")
end_idx = text.find("END OF THE PROJECT GUTENBERG EBOOK")
text = text[start_idx:end_idx]
char_set = set(text)
print("Total Length:", len(text))
print("Unique Characters:", len(char_set))
print("Unique Characters:", char_set)

Total Length: 1112267
Unique Characters: 80
Unique Characters: {',', 'E', ')', 'S', 'R', '&', '!', 'H', 't', '*', 'r', 'Q', 'P', 'B', 'm', 'Z', 'p', 'x', 'd', '/', '-', '2', '?', 'z', '.', ' ', 'T', 'U', 'V', 'l', '“', 'n', '4', 'M', 'a', 'v', 'w', 'u', '8', '=', '’', 'A', 'Y', 'K', '3', ';', 'N', '9', '0', 'i', '(', '1', ':', 'g', '”', 'q', 'f', 'F', '‘', 'L', '5', '\n', 'e', 'o', 'D', 'G', 'I', 'J', 'b', '6', 'c', 's', 'W', 'C', '7', 'j', 'h', 'y', 'O', 'k'}


In [2]:
char_sorted = sorted(char_set)
print(char_sorted)
char2int = {c:i for i,c in enumerate(char_sorted)}
print(char2int)
char_array = np.array(char_sorted)
print(char_array)
text_encoded = np.array([char2int[c] for c in text], dtype = np.int32)
print(text_encoded)
#

print(text[:15], '== Encoding ==>', text_encoded[:15])

for ex in text_encoded[:5]:
  print('{} -> {}'.format(ex, char_array[ex]))

['\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '‘', '’', '“', '”']
{'\n': 0, ' ': 1, '!': 2, '&': 3, '(': 4, ')': 5, '*': 6, ',': 7, '-': 8, '.': 9, '/': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ':': 21, ';': 22, '=': 23, '?': 24, 'A': 25, 'B': 26, 'C': 27, 'D': 28, 'E': 29, 'F': 30, 'G': 31, 'H': 32, 'I': 33, 'J': 34, 'K': 35, 'L': 36, 'M': 37, 'N': 38, 'O': 39, 'P': 40, 'Q': 41, 'R': 42, 'S': 43, 'T': 44, 'U': 45, 'V': 46, 'W': 47, 'Y': 48, 'Z': 49, 'a': 50, 'b': 51, 'c': 52, 'd': 53, 'e': 54, 'f': 55, 'g': 56, 'h': 57, 'i': 58, 'j': 59, 'k': 60, 'l': 61, 'm': 62, 'n': 63, 'o': 64, 'p': 65, 'q': 66, 'r'

In [3]:
import torch
from torch.utils.data import Dataset
seq_len = 40
chunk_size = seq_len + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size)]
print(len(text_chunks))
# print(text_chunks[111])

class TextDataset(Dataset):

    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):

        return len(self.text_chunks)

    def __getitem__(self, idx):

        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()

seq_dataset = TextDataset(torch.tensor(text_chunks))

1112226


/tmp/ipykernel_38031/1526932595.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ./torch/csrc/utils/tensor_new.cpp:253.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


In [4]:
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x): ', repr(''.join(char_array[seq])))
    print('Target (y): ', repr(''.join(char_array[target])))
    print()
    if i == 1:
        break

 Input (x):  'THE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n1'
Target (y):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'

 Input (x):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'
Target (y):  'E MYSTERIOUS ISLAND\n\nby Jules Verne\n\n187'



In [5]:
from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# Building a character-level RNN model

In [6]:
import torch.nn as nn
class CharRNN(nn.Module):
  def __init__(self, vocab_size, embed_dim, rnn_hidden_state) :
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.rnn_hidden_state = rnn_hidden_state
    self.rnn = nn.LSTM(embed_dim, rnn_hidden_state, batch_first=True)
    self.fc = nn.Linear(rnn_hidden_state, vocab_size)

  def forward(self, x, hidden, cell):
    out = self.embedding(x).unsqueeze(1)
    out, (hidden, cell) = self.rnn(out, (hidden, cell))
    out = self.fc(out).reshape(out.size(0), -1)
    return out, hidden, cell

  def init_state(self, batch_size):
    hidden = torch.zeros(1, batch_size, self.rnn_hidden_state)
    cell = torch.zeros(1, batch_size, self.rnn_hidden_state)
    return hidden, cell
#

In [7]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_state = 512
torch.manual_seed(1)
model = CharRNN(vocab_size, embed_dim, rnn_hidden_state)
model

CharRNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
#

In [ ]:
num_epochs = 10000
torch.manual_seed(1)
for epoch in range(num_epochs):
  hidden, cell = model.init_state(batch_size)
  seq_batch, target_batch = next(iter(seq_dl))
  optimizer.zero_grad()
  loss = 0
  for c in range(seq_len):
    pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
    loss += loss_fn(pred, target_batch[:, c])
  loss.backward()
  optimizer.step()
  loss = loss.item() / seq_len

  if (epoch+1) % 500 == 0:
    print(f'Epoch: {epoch}, Loss: {loss}')
#

In [ ]:
from torch.distributions.categorical import Categorical
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])
print('Probabilities:',nn.functional.softmax(logits, dim=1).numpy()[0])

m = Categorical(logits=logits)
sample = m.sample((10,))
print(sample.numpy())
#

Probabilities: [0.33333334 0.33333334 0.33333334]
[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


In [ ]:
def sample(model, starting_str, len_generated_text= 1000, scale_factor = 1):
  encoded_input = torch.tensor([char2int[s] for s in starting_str]).unsqueeze(0)
  encoded_input = torch.reshape(encoded_input, (1, -1))
  generated_str = starting_str
  model.eval()
  hidden,cell = model.init_state(1)
  for c in range(len(starting_str) - 1):
    _,hidden,cell = model(encoded_input[:,c].view(1), hidden, cell)
  last_char = encoded_input[:, -1]
  for i in range(len_generated_text):
    logits, hidden, cell = model(last_char.view(1), hidden, cell)
    logits = torch.squeeze(logits, 0)
    scaled_logits = logits * scale_factor
    m = Categorical(logits=scaled_logits)
    last_char = m.sample()
    generated_str += str(char_array[last_char])
  return generated_str
#

In [ ]:
torch.manual_seed(1)
print(sample(model, starting_str="The Island"))

The Island of material was
sleeping.
The “Bonadventure” had not, occupied a tropica short, was unaccountable,” replied the sailor.

“An it indeed,” and he wedged to draw light it appeared unter, with it will, even their strength light! Neb end to the same aquatic
qualities.

The very agreeable dwentiments was heat, and all the woods.

Threatened Pencroft’s eyes, were only departed, “do not pet one thing.”

“Good!” replied Gideon Spilett and Herbert did not discover six o’clock much as we do. To enter the north wind, but the sea-west,” replied Creek, although there was not
a mixture up the sea.

“Let remain?”

“Yes,” replied Pencroft; “but it was thrown on the coal or making a trouble of mummum!”

“But Captain Harding,” asked Pencroft one, however, they would be included, Were they set off to Ruch a height of turus Captain, but little hope to the shore, cracked towards the danger the moon I think
between the ground, the castaways in the other name of treastic acid, that is to be next da

In [ ]:
torch.manual_seed(1)
# print(sample(model, starting_str="The Island"))
# lets  try adjusting temperature i.e by multiplying logits with a constant temp with a logit (scale factor) to produce scaled logit
print(sample(model, starting_str=("The food"), len_generated_text= 50, scale_factor = 2) )


The food of the lake was strangely confidence, and the boy
